# 🛡️ Fine-tuning PhayaThaiBERT for Scam Detection (KU ScamGuard)

สมุดบันทึกนี้ออกแบบมาสำหรับเทรนโมเดลภาษาไทย **PhayaThaiBERT** (`clicknext/phayathaibert`) แบบ **Multi-Task Learning**:
1. **Task 1 (Risk Classification):** แบ่งระดับความเสี่ยง 3 ระดับ (`normal`, `scam`, `suspicious`)
2. **Task 2 (Category Classification):** จำแนกประเภทกลโกง 10 หมวดหมู่ (`delivery`, `fake_scholarship`, `impersonation`, `job`, `marketplace`, `none`, `otp`, `phishing`, `refund`, `tu`)

---
### 📌 ขั้นตอนการทำงาน
1. ติดตั้ง Dependencies และตรวจสอบ GPU
2. อัปโหลดไฟล์ Dataset (`KU_ScamGuard_dataset_Rezero2.xlsx`)
3. Preprocessing ข้อความ (Clean Text + Masking PII & URLs)
4. เทรนโมเดล Multi-Task PhayaThaiBERT พร้อม Validation Checkpoint
5. ประเมินผลบน Test Set (F1-score, Precision, Recall, Confusion Matrix)
6. ส่งออกโมเดล (`model.safetensors` + config + tokenizer) และ Zip พร้อมนำไปใส่ในโฟลเดอร์ `models/phayathaibert/`

## 1. ตรวจสอบ GPU และติดตั้ง Dependencies

In [ ]:
!nvidia-smi
!pip install -q transformers>=4.44.0 datasets safetensors openpyxl scikit-learn accelerate sentencepiece

In [ ]:
import os
import re
import json
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from safetensors.torch import save_file as save_safetensors
from sklearn.metrics import classification_report, f1_score, accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'Device name: {torch.cuda.get_device_name(0)}')

## 2. อัปโหลดไฟล์ Dataset (`KU_ScamGuard_dataset_Rezero2.xlsx`)

In [ ]:
from google.colab import files

dataset_filename = 'KU_ScamGuard_dataset_Rezero2.xlsx'
if not os.path.exists(dataset_filename):
    print('กรุณาอัปโหลดไฟล์ KU_ScamGuard_dataset_Rezero2.xlsx:')
    uploaded = files.upload()
else:
    print(f'พบไฟล์ {dataset_filename} เรียบร้อยแล้ว')

## 3. โหลดและเตรียมข้อมูล (Load & Inspect Dataset)

In [ ]:
train_df = pd.read_excel(dataset_filename, sheet_name='Train')
val_df = pd.read_excel(dataset_filename, sheet_name='Validation')
test_df = pd.read_excel(dataset_filename, sheet_name='Test')

print(f'Train samples: {len(train_df)}')
print(f'Validation samples: {len(val_df)}')
print(f'Test samples: {len(test_df)}')

# สร้าง Mapping สำหรับ Label และ Category
RISK_LABELS = ['normal', 'scam', 'suspicious']
CATEGORIES = sorted([
    'delivery', 'fake_scholarship', 'impersonation', 'job', 'marketplace',
    'none', 'otp', 'phishing', 'refund', 'tu'
])

risk2id = {label: i for i, label in enumerate(RISK_LABELS)}
id2risk = {i: label for label, i in risk2id.items()}
cat2id = {cat: i for i, cat in enumerate(CATEGORIES)}
id2cat = {i: cat for cat, i in cat2id.items()}

print('\nRisk labels:', risk2id)
print('Categories:', cat2id)

## 4. Text Preprocessing (Clean Text Logic)
แปลง URL, เบอร์โทร, LINE ID, และบัญชีธนาคาร ให้เป็น Token พิเศษเหมือนตอนรัน Production

In [ ]:
URL_RE = re.compile(
    r'(?i)(?:https?://|www\.)\S+|\b[a-z0-9][a-z0-9\-]*(?:\.[a-z0-9\-]+)*\.'
    r'(?:invalid|com|net|org|io|co|me|th|ly|link|xyz|top|info)\b(?:/\S*)?|\bbit\.ly/\S+'
)
LINE_RE = re.compile(
    r'(?i)(?:\b(?:line(?:\s*id)?|ไลน์|ไอดีไลน์)\s*[:：]?\s*)?@[A-Za-z0-9._\-]+'
)
PHONE_RE = re.compile(r'(?<!\d)0\d[\dxX][- ]?[\dxX]{3}[- ]?[\dxX]{3,4}(?!\d)')
BANK_RE = re.compile(r'(?i)\b(?:x{2,}\d{2,}|\d{3}-\d-\d{5}-\d|\d{10,12})\b')

def clean_text(t: str) -> str:
    if not isinstance(t, str):
        t = str(t or '')
    t = URL_RE.sub('[URL]', t)
    t = LINE_RE.sub('[LINE_ID]', t)
    t = PHONE_RE.sub('[PHONE]', t)
    t = BANK_RE.sub('[BANK_ACC]', t)
    return re.sub(r'\s+', ' ', t).strip()

# ตรวจสอบตัวอย่างการ Clean
sample_raw = 'ด่วน! ได้รับสิทธิ์เงินกู้ ทักไลน์ @scam99 หรือโทร 0812345678 คลิก https://fake.link'
print('Raw:    ', sample_raw)
print('Cleaned:', clean_text(sample_raw))

## 5. Tokenizer & Multi-Task Dataset

In [ ]:
BASE_MODEL_NAME = 'clicknext/phayathaibert'
MAX_LEN = 192

print(f'Loading Tokenizer for {BASE_MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

class ScamMultiTaskDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=192):
        self.texts = [clean_text(t) for t in df['text']]
        self.risk_labels = [risk2id[r] for r in df['label']]
        self.cat_labels = [cat2id.get(c, cat2id['none']) for c in df['category']]
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'risk_label': torch.tensor(self.risk_labels[idx], dtype=torch.long),
            'cat_label': torch.tensor(self.cat_labels[idx], dtype=torch.long),
        }

train_dataset = ScamMultiTaskDataset(train_df, tokenizer, max_len=MAX_LEN)
val_dataset = ScamMultiTaskDataset(val_df, tokenizer, max_len=MAX_LEN)
test_dataset = ScamMultiTaskDataset(test_df, tokenizer, max_len=MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## 6. สถาปัตยกรรมโมเดล PhayaThaiBERT Multi-Task

In [ ]:
class PhayaThaiBERTMultiTaskModel(nn.Module):
    def __init__(self, model_name: str, num_risk_labels: int, num_cat_labels: int, dropout: float = 0.15):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.risk_classifier = nn.Linear(h, num_risk_labels)
        self.category_classifier = nn.Linear(h, num_cat_labels)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # ดึง [CLS] token representation
        cls_rep = self.dropout(out.last_hidden_state[:, 0, :])
        risk_logits = self.risk_classifier(cls_rep)
        cat_logits = self.category_classifier(cls_rep)
        return risk_logits, cat_logits

model = PhayaThaiBERTMultiTaskModel(
    model_name=BASE_MODEL_NAME,
    num_risk_labels=len(RISK_LABELS),
    num_cat_labels=len(CATEGORIES)
).to(device)

print('โมเดลถูกสร้างและย้ายเข้าสู่อุปกรณ์เรียบร้อย:', device)

## 7. Hyperparameters, Optimizer & Loss Function

In [ ]:
EPOCHS = 6
LR = 2e-5
WEIGHT_DECAY = 0.01
LAMBDA_CAT = 0.6  # น้ำหนัก Loss ของ Category Classification

# คำนวณ Class weights สำหรับ Risk Level เพื่อเน้น Scam / Suspicious
risk_counts = train_df['label'].value_counts()
total_samples = len(train_df)
risk_weights = [total_samples / (len(RISK_LABELS) * risk_counts[r]) for r in RISK_LABELS]
risk_weights_tensor = torch.tensor(risk_weights, dtype=torch.float).to(device)
print('Risk Weights:', {r: round(w, 2) for r, w in zip(RISK_LABELS, risk_weights)})

criterion_risk = nn.CrossEntropyLoss(weight=risk_weights_tensor)
criterion_cat = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

## 8. วงรอบการเทรน (Training Loop & Validation Evaluation)

In [ ]:
def evaluate(model, loader):
    model.eval()
    all_risk_preds, all_risk_true = [], []
    all_cat_preds, all_cat_true = [], []
    total_val_loss = 0.0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            risk_labels = batch['risk_label'].to(device)
            cat_labels = batch['cat_label'].to(device)

            r_logits, c_logits = model(input_ids, attention_mask)
            l_risk = criterion_risk(r_logits, risk_labels)
            l_cat = criterion_cat(c_logits, cat_labels)
            loss = l_risk + LAMBDA_CAT * l_cat
            total_val_loss += loss.item()

            all_risk_preds.extend(torch.argmax(r_logits, dim=-1).cpu().numpy())
            all_risk_true.extend(risk_labels.cpu().numpy())
            all_cat_preds.extend(torch.argmax(c_logits, dim=-1).cpu().numpy())
            all_cat_true.extend(cat_labels.cpu().numpy())

    macro_f1 = f1_score(all_risk_true, all_risk_preds, average='macro')
    scam_f1 = f1_score(all_risk_true, all_risk_preds, labels=[risk2id['scam']], average='macro')
    cat_f1 = f1_score(all_cat_true, all_cat_preds, average='weighted')
    avg_loss = total_val_loss / len(loader)

    return avg_loss, macro_f1, scam_f1, cat_f1

best_val_f1 = 0.0
best_model_state = None

print('🚀 เริ่มกระบวนการเทรน...')
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_train_loss = 0.0
    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        risk_labels = batch['risk_label'].to(device)
        cat_labels = batch['cat_label'].to(device)

        optimizer.zero_grad()
        r_logits, c_logits = model(input_ids, attention_mask)
        l_risk = criterion_risk(r_logits, risk_labels)
        l_cat = criterion_cat(c_logits, cat_labels)
        loss = l_risk + LAMBDA_CAT * l_cat

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_train_loss += loss.item()

    train_loss = total_train_loss / len(train_loader)
    val_loss, val_macro_f1, val_scam_f1, val_cat_f1 = evaluate(model, val_loader)

    print(f'Epoch {epoch}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Macro-F1: {val_macro_f1:.4f} (Scam F1: {val_scam_f1:.4f}) | Val Cat-F1: {val_cat_f1:.4f}')

    # บันทึก Checkpoint ที่ได้คะแนนดีที่สุด
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        best_model_state = copy.deepcopy(model.state_dict())
        print(f'  🌟 บันทึก Best Model ใหม่ (Val Macro-F1: {best_val_f1:.4f})')

# โหลด state ที่ดีที่สุดกลับมา
model.load_state_dict(best_model_state)
print('\n✅ โหลด Best Model ประจำรอบการทดสอบเรียบร้อยแล้ว')

## 9. ทดสอบประเมินผลบน Test Set (Final Test Evaluation)

In [ ]:
model.eval()
all_r_preds, all_r_true = [], []
all_c_preds, all_c_true = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        r_logits, c_logits = model(input_ids, attention_mask)
        all_r_preds.extend(torch.argmax(r_logits, dim=-1).cpu().numpy())
        all_r_true.extend(batch['risk_label'].numpy())
        all_c_preds.extend(torch.argmax(c_logits, dim=-1).cpu().numpy())
        all_c_true.extend(batch['cat_label'].numpy())

print('=== 📊 รายงานผล Risk Level Classification (Test Set) ===')
print(classification_report(all_r_true, all_r_preds, target_names=RISK_LABELS, digits=4))

print('\n=== 📊 รายงานผล Category Classification (Test Set) ===')
print(classification_report(all_c_true, all_c_preds, target_names=CATEGORIES, digits=4, zero_division=0))

## 10. ส่งออกโมเดลสำหรับ Production (`models/phayathaibert/`)
ส่งออกเป็นรูปแบบ SafeTensors, ScamGuard Config JSON, และ Tokenizer ตามที่ระบบ Inference ต้องการ

In [ ]:
OUTPUT_DIR = 'phayathaibert_scamguard_export'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. เซฟน้ำหนักโมเดลเป็น model.safetensors
safetensors_path = os.path.join(OUTPUT_DIR, 'model.safetensors')
save_safetensors(model.state_dict(), safetensors_path)
print(f'✅ บันทึก SafeTensors: {safetensors_path}')

# 2. บันทึก Tokenizer
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'✅ บันทึก Tokenizer ใน: {OUTPUT_DIR}')

# 3. บันทึก scamguard_config.json
scamguard_config = {
    'base_model': BASE_MODEL_NAME,
    'risk_labels': RISK_LABELS,
    'categories': CATEGORIES,
    'max_len': MAX_LEN,
    't_red': 0.55,
    't_yellow': 0.45,
    'weights_format': 'safetensors',
    'model_architecture': 'PhayaThaiBERTMultiTaskModel'
}

config_path = os.path.join(OUTPUT_DIR, 'scamguard_config.json')
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(scamguard_config, f, ensure_ascii=False, indent=2)
print(f'✅ บันทึก Config: {config_path}')

# 4. สรุปรายการไฟล์ที่ส่งออก
print('\nรายการไฟล์ที่พร้อมนำไปใช้งาน:')
for fname in os.listdir(OUTPUT_DIR):
    fsize = os.path.getsize(os.path.join(OUTPUT_DIR, fname))
    print(f'  - {fname} ({fsize:,} bytes)')

## 11. บีบอัดไฟล์ Zip และดาวน์โหลดกลับมาใส่ในโฟลเดอร์โปรเจกต์
เมื่อดาวน์โหลดไฟล์ `phayathaibert_scamguard.zip` แล้ว ให้นำเนื้อหาด้านในไปแตกไฟล์ใส่ในโฟลเดอร์:
`d:\Scammer\Scammer_local\models\phayathaibert\`

In [ ]:
import shutil
from google.colab import files

zip_filename = 'phayathaibert_scamguard'
shutil.make_archive(zip_filename, 'zip', OUTPUT_DIR)
print(f'บีบอัดเรียบร้อย: {zip_filename}.zip')

# ดาวน์โหลดอัตโนมัติ
files.download(f'{zip_filename}.zip')